In [8]:
import pandas as pd
import numpy as np

# read in all our data
df = pd.read_csv('shit.csv', encoding='latin1')
pd.set_option("display.max_columns", 50)

# set seed for reproducibility
np.random.seed(0) 

In [ ]:
# Strip hidden whitespace from all column names
df.columns = df.columns.str.strip()

# Clean numerical anomalies before casting
df['SalaryUSD'] = df['SalaryUSD'].astype(str).str.replace(',', '').str.replace(' ', '')
# DatabaseServers has a few "+" characters in the strings (e.g. "500+")
df['DatabaseServers'] = df['DatabaseServers'].astype(str).str.replace('+', '')

# Convert literal "Not Asked" strings to actual Pandas NaNs across the whole dataframe
df.replace('Not Asked', np.nan, inplace=True)

# Drop columns with >90% missing data that likely aren't useful
df.drop(columns=['MonthsUnemployed', 'DataAndServersAreIn'], inplace=True)

# Fill missing categorical data with an explicit label
categorical_cols = df.select_dtypes(include=['object']).columns
df[categorical_cols] = df[categorical_cols].fillna('Missing_Or_Not_Asked')

# Explicit Type Casting Dictionary
type_mapping = {
    # Numerics
    'SalaryUSD': 'float64',
    'Survey Year': 'Int64',
    'YearsWithThisDatabase': 'Int64',
    'YearsWithThisTypeOfJob': 'Int64',
    'DatabaseServers': 'float64', # Floats handle NaNs natively
    'MonthsUnemployed': 'float64',
    
    # Categoricals (Low Cardinality)
    'EmploymentStatus': 'category',
    'JobTitle': 'category',
    'HowManyCompanies': 'category',
    'OtherPeopleOnYourTeam': 'category',
    'PopulationOfLargestCityWithin20Miles': 'category',
    'EmploymentSector': 'category',
    'CareerPlansThisYear': 'category',
    'Gender': 'category',
    
    # Explicit Strings (High Cardinality or Multi-value)
    'Country': 'string',
    'PrimaryDatabase': 'string',
    'OtherDatabases': 'string',
    'DataAndServersAreIn': 'string'
}

# Apply Date and Boolean transformations first
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df['ManageStaff'] = df['ManageStaff'].map({'Yes': True, 'No': False}).astype('boolean')

# Apply the rest of the mapping
for col, dtype in type_mapping.items():
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce') if 'float' in dtype or 'Int' in dtype else df[col].astype(dtype)

# Drop redundant or garbage columns
df.drop(columns=['PostalCode', 'Counter'], inplace=True, errors='ignore')

# Verify the new state
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14869 entries, 0 to 14868
Data columns (total 18 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   Survey Year                           14869 non-null  int64         
 1   Timestamp                             14869 non-null  datetime64[ns]
 2   SalaryUSD                             14869 non-null  int64         
 3   Country                               14869 non-null  string        
 4   PrimaryDatabase                       14869 non-null  string        
 5   YearsWithThisDatabase                 14869 non-null  int64         
 6   OtherDatabases                        14869 non-null  string        
 7   EmploymentStatus                      14869 non-null  category      
 8   JobTitle                              14869 non-null  category      
 9   ManageStaff                           14869 non-null  boolean       
 10

In [ ]:
# Define the Top 20 most frequent databases as your strict schema
top_dbs = [
    'Microsoft SQL Server', 'MySQL/MariaDB', 'Oracle', 'Microsoft Access', 
    'PostgreSQL', 'Azure SQL DB (any flavor)', 'Amazon RDS (any flavor)', 
    'Azure SQL DB', 'MongoDB', 'SQLite', 'DB2', 'CosmosDB', 'Teradata', 
    'Elasticsearch', 'Redis', 'Cassandra', 'SAP', 'Snowflake', 'SAP HANA', 'Informix'
]

# Function to clean and bucket the comma-separated strings
def consolidate_databases(db_string):
    if pd.isna(db_string) or db_string == 'Missing_Or_Not_Asked':
        return np.nan
        
    # Split the string by commas and strip whitespace
    db_list = [db.strip() for db in str(db_string).split(',') if db.strip()]
    
    clean_list = []
    for db in db_list:
        # If the database is in our Top 20, keep it exactly as is
        if db in top_dbs:
            clean_list.append(db)
        # Otherwise, bucket it into "Uncommon"
        else:
            clean_list.append('Uncommon')
            
    # Return as a clean comma-separated string, removing duplicates
    return ', '.join(list(set(clean_list)))

# Apply the consolidation logic
df['OtherDatabases'] = df['OtherDatabases'].apply(consolidate_databases)

# Now run get_dummies on the clean data
other_db_encoded = df['OtherDatabases'].str.get_dummies(sep=', ')
other_db_encoded = other_db_encoded.add_prefix('OtherDB_')
other_db_encoded = other_db_encoded.astype('boolean')

# Join and drop original
df = pd.concat([df, other_db_encoded], axis=1)
df.drop(columns=['OtherDatabases'], inplace=True)

# Print the resulting schema dimensions
print("New database columns count:", len(other_db_encoded.columns))
print(list(other_db_encoded.columns))

New database columns count: 21
['OtherDB_Amazon RDS (any flavor)', 'OtherDB_Azure SQL DB', 'OtherDB_Azure SQL DB (any flavor)', 'OtherDB_Cassandra', 'OtherDB_CosmosDB', 'OtherDB_DB2', 'OtherDB_Elasticsearch', 'OtherDB_Informix', 'OtherDB_Microsoft Access', 'OtherDB_Microsoft SQL Server', 'OtherDB_MongoDB', 'OtherDB_MySQL/MariaDB', 'OtherDB_Oracle', 'OtherDB_PostgreSQL', 'OtherDB_Redis', 'OtherDB_SAP', 'OtherDB_SAP HANA', 'OtherDB_SQLite', 'OtherDB_Snowflake', 'OtherDB_Teradata', 'OtherDB_Uncommon']


In [11]:
print(df.info(memory_usage='deep'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14869 entries, 0 to 14868
Data columns (total 38 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   Survey Year                           14869 non-null  int64         
 1   Timestamp                             14869 non-null  datetime64[ns]
 2   SalaryUSD                             14869 non-null  int64         
 3   Country                               14869 non-null  string        
 4   PrimaryDatabase                       14869 non-null  string        
 5   YearsWithThisDatabase                 14869 non-null  int64         
 6   EmploymentStatus                      14869 non-null  category      
 7   JobTitle                              14869 non-null  category      
 8   ManageStaff                           14869 non-null  boolean       
 9   YearsWithThisTypeOfJob                14869 non-null  int64         
 10

In [12]:
# Check min, max, and mean to ensure the parsing didn't create bizarre outliers
print(df[['SalaryUSD', 'DatabaseServers', 'Survey Year']].describe())

          SalaryUSD  DatabaseServers   Survey Year
count  1.486900e+04     13122.000000  14869.000000
mean   1.007982e+05       494.586420   2020.281122
std    6.151424e+04     15905.497457      2.873474
min    1.300000e+01         0.000000   2017.000000
25%    6.900000e+04         6.000000   2018.000000
50%    9.600000e+04        20.000000   2020.000000
75%    1.250000e+05        80.000000   2022.000000
max    1.850000e+06   1000000.000000   2026.000000


In [13]:
# Sum up the True values in all the new OtherDB_ columns
encoded_dbs = df.filter(like='OtherDB_')
print("Total True values per database category:")
print(encoded_dbs.sum().sort_values(ascending=False))

Total True values per database category:
OtherDB_Microsoft SQL Server         5914
OtherDB_MySQL/MariaDB                3653
OtherDB_Oracle                       3558
OtherDB_Microsoft Access             3393
OtherDB_PostgreSQL                   2928
OtherDB_Azure SQL DB (any flavor)    2522
OtherDB_Amazon RDS (any flavor)      1535
OtherDB_Azure SQL DB                 1361
OtherDB_MongoDB                      1317
OtherDB_SQLite                       1065
OtherDB_Uncommon                     1026
OtherDB_DB2                          1022
OtherDB_CosmosDB                      386
OtherDB_Teradata                      290
OtherDB_Elasticsearch                 270
OtherDB_Redis                         248
OtherDB_Cassandra                     227
OtherDB_SAP                           224
OtherDB_Snowflake                     145
OtherDB_SAP HANA                      127
OtherDB_Informix                       69
dtype: Int64


In [5]:
# Look at row 4, which we know from earlier had a massive list of databases
print("\n--- Spot Check: Row 4 ---")
row = df.iloc[4]

# Check standard categories
print(f"Salary: {row['SalaryUSD']} | Type: {type(row['SalaryUSD'])}")
print(f"Job Title: {row['JobTitle']} | Type: {type(row['JobTitle'])}")

# See exactly which databases flipped to True for this specific person
print("\nDatabases flagged as True for this row:")
print(row[row.index.str.startswith('OtherDB_') & (row == True)])


--- Spot Check: Row 4 ---
Salary: 176000 | Type: <class 'numpy.int64'>
Job Title: Engineer | Type: <class 'str'>

Databases flagged as True for this row:
Series([], Name: 4, dtype: object)


In [14]:
# look at a few rows of the df file. I can see a handful of missing data already!
df.sample(5)
# Survey Year
# Timestamp - 
# SalaryUSD - salary in usd
# Country 
# PostalCode
# PrimaryDatabase


,Survey Year,Timestamp,SalaryUSD,Country,PrimaryDatabase,YearsWithThisDatabase,EmploymentStatus,JobTitle,ManageStaff,YearsWithThisTypeOfJob,HowManyCompanies,OtherPeopleOnYourTeam,DatabaseServers,PopulationOfLargestCityWithin20Miles,EmploymentSector,CareerPlansThisYear,Gender,OtherDB_Amazon RDS (any flavor),OtherDB_Azure SQL DB,OtherDB_Azure SQL DB (any flavor),OtherDB_Cassandra,OtherDB_CosmosDB,OtherDB_DB2,OtherDB_Elasticsearch,OtherDB_Informix,OtherDB_Microsoft Access,OtherDB_Microsoft SQL Server,OtherDB_MongoDB,OtherDB_MySQL/MariaDB,OtherDB_Oracle,OtherDB_PostgreSQL,OtherDB_Redis,OtherDB_SAP,OtherDB_SAP HANA,OtherDB_SQLite,OtherDB_Snowflake,OtherDB_Teradata,OtherDB_Uncommon
14530,2017,2017-01-05 11:17:51,125000,United States,Microsoft SQL Server,10,Full time employee,"Developer: Business Intelligence (SSRS, PowerB...",True,10,Missing_Or_Not_Asked,3,12.0,Missing_Or_Not_Asked,Private business,Missing_Or_Not_Asked,Missing_Or_Not_Asked,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
11745,2018,2017-12-12 11:27:40,95000,United States,Microsoft SQL Server,16,Full time employee,DBA (Production Focus - build & troubleshoot s...,False,7,Missing_Or_Not_Asked,3,200.0,Missing_Or_Not_Asked,Non-profit,"Stay with the same employer, same role",Male,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
3460,2023,2022-11-22 09:17:50,100000,United States,Microsoft SQL Server,22,Full time employee,"Developer: App code (C#, JS, etc)",False,6,4,4,1.0,300K-1M (large city),Private business,"Stay with the same employer, same role",Male,False,False,True,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False
12674,2017,2017-01-09 11:08:59,65000,United States,Microsoft SQL Server,4,Full time employee,Analyst,False,3,Missing_Or_Not_Asked,Missing_Or_Not_Asked,55.0,Missing_Or_Not_Asked,Private business,Missing_Or_Not_Asked,Missing_Or_Not_Asked,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False
9894,2018,2017-12-20 12:03:52,155000,United States,Microsoft SQL Server,12,Full time employee,Manager,True,10,Missing_Or_Not_Asked,1,8.0,Missing_Or_Not_Asked,Private business,"Stay with the same employer, but change roles",Male,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False


In [7]:
# Save with schema and compression intact
df.to_parquet('data_clean.parquet', index=False)

In [1]:
import pandas as pd

# 1. Load the Parquet file (Timestamp dtype is automatically preserved)
df = pd.read_parquet("data_clean.parquet")

# 2. Sort the entire dataframe sequentially by date
# Ensure it is sorted from oldest to newest
df_sorted = df.sort_values(by="Timestamp", ascending=True).reset_index(drop=True)

# 3. Calculate the row index for the 90% cutoff
cutoff_index = int(len(df_sorted) * 0.9)

# 4. Split the data using integer location (.iloc)
train_df = df_sorted.iloc[:cutoff_index]      # The oldest 90%
test_df = df_sorted.iloc[cutoff_index:]       # The most recent 10%

# Verify the dates to ensure the split was correct
print(f"Train End Date: {train_df['Timestamp'].max()}")
print(f"Test Start Date: {test_df['Timestamp'].min()}")

Train End Date: 2024-12-19 05:04:38
Test Start Date: 2024-12-19 05:50:04


In [2]:
train_df.to_parquet('data_train_clean.parquet', index=False)
test_df.to_parquet('data_inf_clean.parquet', index=False)

In [33]:
df.to_csv(
    'data_clean.csv',
    index=False,
    encoding='utf-8',
    date_format='%Y-%m-%d %H:%M:%S'
)

In [15]:
print(df.shape)
print(df.index)
print(df.columns)

(14869, 38)
RangeIndex(start=0, stop=14869, step=1)
Index(['Survey Year', 'Timestamp', 'SalaryUSD', 'Country', 'PrimaryDatabase',
       'YearsWithThisDatabase', 'EmploymentStatus', 'JobTitle', 'ManageStaff',
       'YearsWithThisTypeOfJob', 'HowManyCompanies', 'OtherPeopleOnYourTeam',
       'DatabaseServers', 'PopulationOfLargestCityWithin20Miles',
       'EmploymentSector', 'CareerPlansThisYear', 'Gender',
       'OtherDB_Amazon RDS (any flavor)', 'OtherDB_Azure SQL DB',
       'OtherDB_Azure SQL DB (any flavor)', 'OtherDB_Cassandra',
       'OtherDB_CosmosDB', 'OtherDB_DB2', 'OtherDB_Elasticsearch',
       'OtherDB_Informix', 'OtherDB_Microsoft Access',
       'OtherDB_Microsoft SQL Server', 'OtherDB_MongoDB',
       'OtherDB_MySQL/MariaDB', 'OtherDB_Oracle', 'OtherDB_PostgreSQL',
       'OtherDB_Redis', 'OtherDB_SAP', 'OtherDB_SAP HANA', 'OtherDB_SQLite',
       'OtherDB_Snowflake', 'OtherDB_Teradata', 'OtherDB_Uncommon'],
      dtype='object')


In [16]:
print(df.dtypes)

Survey Year                                      int64
Timestamp                               datetime64[ns]
SalaryUSD                                        int64
Country                                 string[python]
PrimaryDatabase                         string[python]
YearsWithThisDatabase                            int64
EmploymentStatus                              category
JobTitle                                      category
ManageStaff                                    boolean
YearsWithThisTypeOfJob                           int64
HowManyCompanies                              category
OtherPeopleOnYourTeam                         category
DatabaseServers                                float64
PopulationOfLargestCityWithin20Miles          category
EmploymentSector                              category
CareerPlansThisYear                           category
Gender                                        category
OtherDB_Amazon RDS (any flavor)                boolean
OtherDB_Az

In [17]:
df.info()
df.count()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14869 entries, 0 to 14868
Data columns (total 38 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   Survey Year                           14869 non-null  int64         
 1   Timestamp                             14869 non-null  datetime64[ns]
 2   SalaryUSD                             14869 non-null  int64         
 3   Country                               14869 non-null  string        
 4   PrimaryDatabase                       14869 non-null  string        
 5   YearsWithThisDatabase                 14869 non-null  int64         
 6   EmploymentStatus                      14869 non-null  category      
 7   JobTitle                              14869 non-null  category      
 8   ManageStaff                           14869 non-null  boolean       
 9   YearsWithThisTypeOfJob                14869 non-null  int64         
 10

Survey Year                             14869
Timestamp                               14869
SalaryUSD                               14869
Country                                 14869
PrimaryDatabase                         14869
YearsWithThisDatabase                   14869
EmploymentStatus                        14869
JobTitle                                14869
ManageStaff                             14869
YearsWithThisTypeOfJob                  14869
HowManyCompanies                        14869
OtherPeopleOnYourTeam                   14869
DatabaseServers                         13122
PopulationOfLargestCityWithin20Miles    14869
EmploymentSector                        14869
CareerPlansThisYear                     14869
Gender                                  14869
OtherDB_Amazon RDS (any flavor)         14869
OtherDB_Azure SQL DB                    14869
OtherDB_Azure SQL DB (any flavor)       14869
OtherDB_Cassandra                       14869
OtherDB_CosmosDB                  

In [18]:
# get the number of missing data points per column
missing_values_count = df.isnull().sum()

# look at the # of missing points in the first ten columns
missing_values_count[:]

Survey Year                                0
Timestamp                                  0
SalaryUSD                                  0
Country                                    0
PrimaryDatabase                            0
YearsWithThisDatabase                      0
EmploymentStatus                           0
JobTitle                                   0
ManageStaff                                0
YearsWithThisTypeOfJob                     0
HowManyCompanies                           0
OtherPeopleOnYourTeam                      0
DatabaseServers                         1747
PopulationOfLargestCityWithin20Miles       0
EmploymentSector                           0
CareerPlansThisYear                        0
Gender                                     0
OtherDB_Amazon RDS (any flavor)            0
OtherDB_Azure SQL DB                       0
OtherDB_Azure SQL DB (any flavor)          0
OtherDB_Cassandra                          0
OtherDB_CosmosDB                           0
OtherDB_DB

In [ ]:
gh = df["Gender"].isnull()
result_df= gh.loc[gh==True]
result_df
df.loc[4738]

4738    True
Name: Gender, dtype: bool